<div class='heading'>
    <div style='float:left;'><h1>CPSC 8810 Machine Learning for Graphs</h1></div>
     <img style="float: right; padding-right: 10px" width="100" src="https://raw.githubusercontent.com/bsethwalker/clemson-cs4300/main/images/clemson_paw.png"> </div>
     </div>

**Clemson University**<br>
**Fall 2025**<br>
**Instructor(s):** Aaron Masino <br>

## Homework 1: Network Fundamentals
This homework is intended to assess your knowledge of core concepts related to network representations and metrics, algorithms for computing network metrics and their computational complexity, and elements of the Python NetworkX library for as introduced during the in-class lectures and labs. You may wish to refer to the course lectures and labs while completing this assignment. 

**Unless otherwise noted in the problem instructions you may use any of the following Python libraries to complete the exercises:**
- numpy, scipy
- scikit-learn
- matplotlib, seaborn, pygraphviz
- PyTorch, PyTorch Geometric
- NetworkX


In [ ]:
# imports
import networkx as nx
import matplotlib.pyplot as plt
import numpy as np
import time
from torch_geometric.utils import to_networkx

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.metrics import (classification_report, confusion_matrix, 
                           roc_curve, auc)
import seaborn as sns

RANDOM_STATE = 123456

# Helper Functions
The functions in the following cell are helper methods that will be used in exercises below. These will be noted when needed.

In [ ]:
##################################
#### DO NOT MODIFY THIS CODE #####
##################################
def network_diameter(G):
    try:
        if nx.is_connected(G):
            diameter = nx.diameter(G)
        else:
            # For disconnected graphs, use the maximum diameter of connected components
            diameters = []
            for component in nx.connected_components(G):
                subgraph = G.subgraph(component)
                if len(component) > 1:  # Avoid single-node components
                    diameters.append(nx.diameter(subgraph))
            diameter = max(diameters) if diameters else 0
    except:
        diameter = 0
    return diameter

def generate_undirected_adj_matrix(n, p=0.2, rs=RANDOM_STATE):
    """Generate a random adjacency matrix for an undirected graph with n nodes and edge probability p."""
    G = nx.gnp_random_graph(n, p, directed=False, seed=rs)
    return nx.to_numpy_array(G, dtype=int)

def convert_adj_matrix_to_adj_list(adj_matrix):
    """Convert an adjacency matrix to an adjacency list."""
    adj_list = {}
    for i in range(adj_matrix.shape[0]):
        adj_list[i] = list(np.where(adj_matrix[i] == 1)[0].tolist())
    return adj_list

def generate_undirected_adj_list(n, p=0.2, rs=RANDOM_STATE):
    """Generate a random adjacency list for an undirected graph with n nodes and edge probability p."""
    adj_matrix = generate_undirected_adj_matrix(n, p, rs)
    return convert_adj_matrix_to_adj_list(adj_matrix)

def simulate_udirected_graph(num_nodes, p=0.2, rs=RANDOM_STATE):
    """Generate a random undirected graph with n nodes and edge probability p."""
    G = nx.from_dict_of_lists(generate_undirected_adj_list(num_nodes, p, rs))
    return G

def generate_simulated_graphs(num_graphs, num_nodes, p_vals=[0.2], initial_seed=RANDOM_STATE):
    dataset = []
    labels = []
    cnt = 0
    for idx, p in enumerate(p_vals):
        for _ in range(num_graphs):  # Generate 10 graphs for each probability
            dataset.append(simulate_udirected_graph(num_nodes, p, rs=initial_seed+cnt))
            labels.append(idx)
            cnt += 1
    class_names = [f'C{_}' for _ in range(len(np.unique(labels)))]  # Create class names based on unique labels
    return dataset, np.array(labels), class_names

---

# Assertion Tests
There are assertion tests, e.g. `assert x==y`, throughout the assignment. You may execute these code cells to verify that your implementations are working correctly. However, **please do not modify the asseration test code**.

# Exercise 1: Node In-degree Computional Complexity Analysis (5 points) 
In this exercise, you are asked to empirically compare the computational complexity of two different algorithms for computing the in-degree of all nodes in a network when using an adjacency matrix representation or an adjacency list representation. The exercise has three parts:
* (a)(1 point) Implement a method for computing the in-degree of nodes in a network given the adjacency matrix 
* (b)(1 point) Implement a method for computing the in-degree of nodes in a network given the adjacency list 
* (c)(3 points) Implement code to empirically analyze the computational complexity in terms of time of the two methods. This analysis should:

### 1(a). In-degree method using adjacency matrix
In the code cell below, implement the method `in_degree_adj_matrix`. Assume the method input is a 2-D numpy array representing the graph adjacency matrix. The method should return a numpy array of length N (the number of nodes in the graph) where the position n in the array corresponds to node n in the adjacency matrix and the value is the in-degree of node n. RECALL: The i,j element of the adjacency matrix for a direct network is equal to 1 if there is a directed edge from node j to node i.

#### **Do NOT use NetworkX in your solution**

In [ ]:
def in_degree_adj_matrix(ajd_mtrx):
    ### YOUR CODE HERE ###
    # DO NOT USE NETWORKX
    pass

# Try an example
A = generate_undirected_adj_matrix(10, p=0.2)
print(in_degree_adj_matrix(A))

In [ ]:
# Create assertion to test the function
A = np.array([[1,1,1],[0,0,0], [1,0,1]])
assert in_degree_adj_matrix(A).tolist() == [3.0, 0.0, 2.0], "Test failed!"

### 1(b) In-degree method using adjacency list
In the code cell below, implement the method `in_degree_adj_list`. Assume the method input is dictionary with integer keys representing the node ID and list values where the elements of the list are integers indicating nodes that have a directed edge into the key node. For example, `{0:[2,5]}` would indicate that there is a directed edge from node 2 to node 0 and a directed edge from node 5 to node 0. The method should return a numpy array of length N (the number of nodes in the graph) where the position n in the array corresponds to node n in the adjacency list and the value is the in-degree of node n. 

#### **Do NOT use NetworkX in your solution**

In [ ]:
def in_degree_adj_list(adj_list):
    ### YOUR CODE HERE ###
    # DO NOT USE NETWORKX
    pass

# Try an example
Alist = generate_undirected_adj_list(10, p=0.2)
print(in_degree_adj_list(Alist))

In [ ]:
# Create assertion to test the function
A = {0: [0,1,2], 1: [], 2: [0,2]}
assert in_degree_adj_list(A).tolist() == [3.0, 0.0, 2.0], "Test failed!"

### 1(c) Empirical analysis of time complexity for in-degree methods

In the code cell below:
- complete the implemenation of the `computation_times` method. 
   - It takes as input:
      - `graph`: an adjacency matrix or adjacency list
      - `func`: a function object for a function that can operate on the `graph` input
      - `K`: the number of repitions
   - It should return a list of `K` runtimes (see Python time.perf method) of `func(graph)`. 
- Conduct an empirical analysis of the in degree algorithms using the `computation_times` method.
   - The `adj_*` lists and a `for` loop have been created for you. Use these to store the run times for the `in_degree_adj_matrix` and `in_degree_adj_list` methods you created previously. The input graph adjacency matrix and adjacency list are already created in the `for` loop.
- Create plot showing the mean computation time of each method as a function of the graph node counts. The plot should include error bars corresponding to the standard deviation of the runtimes for each graph size. You should expect to see that as the number of nodes increases the computation time for the adjacency matrix based method grows linearly, while the adjacency list based method is nearly constant time.

In [ ]:
def computation_times(graph, func, K):
    ### YOUR CODE HERE ###
    pass

# Conduct empirical analysis of time complexity for in-degree methods
adj_matr_means = []
adj_matr_stdevs = []
adj_list_means = []
adj_list_stdevs = []
K = 200  # number of repetitions for each graph size
sizes = [10, 50, 100, 200, 300, 400, 500]
for N in sizes:
    adj_mtrx = generate_undirected_adj_matrix(N, p=0.2)
    adj_list = convert_adj_matrix_to_adj_list(adj_mtrx)
    ### YOUR CODE HERE ###
    
# Create a plot to visualize the results
# YOUR CODE HERE

--- 
# Exercise 2: Local Clustering Coefficient Implementation  (5 points)
In this exercise, you are asked to implement the algorithm for local clustering coefficient computation and compare its computation time to the NetworkX implementation. This exercise has three
 parts:
- (a)(2 points) Implement the `local_clustering_coeficient` algorithm
- (b)(1 point) Implement the `local_clustering_all` method
- (b)(2 points) Compare the computation time of your implementation with the NetworkX implementation

### 2(a) Local Clustering Coeficient Implementation for 1 Node
In the code cell below, implement the local clustering coefficient algorithm in the `local_clustering_coefficient` method. The algorithm takes as input:
- `graph` - adjacency list dictionary of the graph
- `node` - the node for which the local clustering coefficient is to be calculated

The function should return the local clustering coefficient for `node`

#### **Do NOT use NetworkX in your solution**

In [ ]:
# Exercise 2(a): Local Clustering Coefficient Implementation  (2 points)
def local_clustering_coefficient(graph, node):
    """Compute the local clustering coefficient for a specific node in the graph."""
    # YOUR CODE HERE
    # DO NOT USE NETWORKX
    pass

# Try an example
Alist = generate_undirected_adj_list(10, p=0.2)
print(local_clustering_coefficient(Alist, 0))

In [ ]:
# assertion tests
A_list = generate_undirected_adj_list(10, p=0.2)
for k in A_list.keys():
    assert local_clustering_coefficient(A_list, k) == nx.clustering(nx.from_dict_of_lists(A_list), k), f"Local clustering coefficient for node {k} does not match NetworkX result"

### 2(b) Local Clustering Coeficient Implementation for All Nodes
In the code cell below, implement the `local_clustering_coefficient_all` method. The method takes as input:
- `graph` - adjacency list dictionary of the graph

The function should return a dictionary where keys are the node id and the values are the local clustering coeficient for the node. Your function should use the `local_clustering_coefficient` method created in the previous exercise.

#### **Do NOT use NetworkX in your solution**

In [ ]:
# Exercise 2(b): Local Clustering Coefficient Implementation  (2 points)
def local_clustering_coefficient_all(graph):
    """Compute the local clustering coefficient for all nodes in the graph."""
    # YOUR CODE HERE
    # DO NOT USE NETWORKX
    pass

# Try an example
Alist = generate_undirected_adj_list(10, p=0.2)
print(local_clustering_coefficient_all(Alist))

In [ ]:
# assertion tests
A_list = generate_undirected_adj_list(10, p=0.2)
coefficients = local_clustering_coefficient_all(A_list)
# compare with NetworkX implementation
G = nx.from_dict_of_lists(A_list)
nx_coefficients = nx.clustering(G)
for k,v in coefficients.items():
    assert np.isclose(v, nx_coefficients[k]), f"Local clustering coefficient for node {k} does not match NetworkX result: {v} vs {nx_coefficients[k]}"

### 2(c) Empirical analysis of time complexity for in-degree methods

In the code cell below:
- Conduct an empirical analysis of the run tims of your local clustering coefficient implementation and the Networkx implementation. You should be able to use your `computation_times` method from problem 1.
   - The `your_impl_*` and `nx_impl_*` lists and a `for` loop have been created for you. Use these to store the run times for the `local_clustering_coefficient_all` and `nx.clustering` methods. The input graph adjacency matrix and adjacency list are already created in the `for` loop.
- Create a plot showing the mean computation time of each method as a function of the graph node counts. The plot should include error bars corresponding to the standard deviation of the runtimes for each graph size. You should expect to see that as the number of nodes increases the computational time both methods grows quadratically. You may also see that the Networkx method performs slightly better. This is one of the advantages of using well curated libraries that leverage varioius optimization schemes (e.g., vectorization, C/C++ libraries).

In [ ]:
your_impl_means = []
your_impl_stdevs = []
nx_impl_means = []
nx_impl_stdevs = []
K = 200  # number of repetitions for each graph size
sizes = [10, 50, 75, 100, 150, 200]
for N in sizes:
    adj_list = convert_adj_matrix_to_adj_list(generate_undirected_adj_matrix(N, p=0.2))
    G = nx.from_dict_of_lists(adj_list)
    # YOUR CODE HERE

# Create a plot to visualize the results
# YOUR CODE HERE

---

# Exercise 3: Construct a ML-Ready Feature Matrix for Graphs (5 points)
In this exercise, you are asked to implement methods that will be used to construct a feature matrix representing graph samples that can be used to train a machine learning model using scikit-learn. The D x p feature matrix will contain D rows where each row contains p features for a graph. The features will be taken from the graph metrics discussed in class and computed using Networkx methods:
- (a)(1 point) Implement the `compute_node_feature_mean` method
- (b)(2 points) Implement the `create_features` method
- (b)(2 points) Implement the `create_feature_matrix` method

### 3(a) Compute Mean of Node Level features
In the code cell below, implement the `compute_node_feature_mean` method. The method takes as input:
- `G` - a Networkx Graph object
- `func` - a function object for a Networkx function that computes a value for every node

The function should return the mean value of the values returned by `func(G)`.

In [ ]:
def compute_node_feature_mean(G, func):
    #### YOUR CODE HERE ####
    pass

# Try an example
G = simulate_udirected_graph(100, p=0.2)
print("Mean degree centrality:", compute_node_feature_mean(G, nx.degree_centrality))

In [ ]:
# assertion test
assert compute_node_feature_mean(G, nx.degree_centrality) == np.mean(list(nx.degree_centrality(G).values())), "Mean degree centrality does not match NetworkX result"

### 3(b) Create Features for One Graph
In the code cell below, implement the `create_features` method. The method takes as input:
- `G` - a Networkx Graph object
- `metrics` - a dictionary where keys are a Networkx method name and values are tuples of `(func, node_or_graph)` where `func` is a function object and `node_or_graph` is a string with allowed values `['node', 'graph']` indicating if `func` is graph level metric (e.g., diameter) or a node level metric (e.g., degree centrality). 

The function should return a numpy array, `features` that contains 1 value for each value in the `metrics`. Specificaly, if `func` is a node level metric the entry in `features` should be the mean value (use the `compute_node_feature_mean` function you created above), and if `func` is a graph level metric the entry in `features` should be the value returned by `func(G)`.

In [ ]:
def create_features(G, metrics):
    ### YOUR CODE HERE ###
    features = None
    
    return features

# Try an example
metrics = {
    'degree': (nx.degree_centrality, 'node'),
    'density': (nx.density, 'graph')
}
G = simulate_udirected_graph(100, p=0.2)
print(create_features(G, metrics))

In [ ]:
G = simulate_udirected_graph(100, p=0.2)
metrics = {
    'degree': (nx.degree_centrality, 'node'),
    'density': (nx.density, 'graph')
}
assert create_features(G, metrics).tolist() == [compute_node_feature_mean(G, nx.degree_centrality), nx.density(G)], "Feature vector does not match expected values"

### 3(c) Create Features for One Graph
In the code cell below, implement the `create_feature_matrix` method. The method takes as input:
- `graphs` - a list of Networkx Graph objects
- `metrics` - a dictionary where keys are a Networkx method name and values are tuples of `(func, node_or_graph)` where `func` is a function object and `node_or_graph` is a string with allowed values `['node', 'graph']` indicating if `func` is graph level metric (e.g., diameter) or a node level metric (e.g., degree centrality). 

The function should return a numpy array, `feature_matrix` that is of shape `(len(graphs),len(metrics))`. Earch row in the array should contain 1 value for each value in the `metrics`. Specificaly, if `func` is a node level metric the value should be the mean value over the nodes in the graph, and if `func` is a graph level metric the entry should be the value returned by `func(G)`. You should use the `create_features` method you created above.

In [ ]:
def create_feature_matrix(dataset, metrics):
    ### YOUR CODE HERE
    feature_matrix = None

    return feature_matrix

# Try an example
dataset, _, _ = generate_simulated_graphs(5, 100, p_vals=[0.2])
metrics = {
    'degree': (nx.degree_centrality, 'node'),
    'density': (nx.density, 'graph')
}
print(create_feature_matrix(dataset, metrics))

In [ ]:
# assertion test
# Try an example
dataset, _, _ = generate_simulated_graphs(2, 100, p_vals=[0.2])
metrics = {
    'degree': (nx.degree_centrality, 'node'),
    'density': (nx.density, 'graph')
}
fm = create_feature_matrix(dataset, metrics)
assert fm.shape == (2, 2), "Feature matrix shape does not match expected (2, 2)"
assert fm[0, 0] == compute_node_feature_mean(dataset[0], nx.degree_centrality), "First row, first column does not match expected value"
assert fm[0, 1] == nx.density(dataset[0]), "First row, second column does not match expected value"
assert fm[1, 0] == compute_node_feature_mean(dataset[1], nx.degree_centrality), "Second row, first column does not match expected value"
assert fm[1, 1] == nx.density(dataset[1]), "Second row, second column does not match expected value"

---

# Exercise 4: Build And Evaluate a Graph Classification Model with Scikit-Learn (10 points)
In this exercise, you are asked to develop and evaluate a machine learning model to infer the label of an input graph. Here, we will represent the graph with engineered metrics building on the code you've create above. We will first create a simulted dataset where the graphs are generated randomly. The graph class labels are assigned based on the probability of any two nodes in the graph being connected by an edge. We will then compute the engineered features on the simulated dataset. Next, we will train a Random Forest classifier to infer graph membership. Finally, we will evaluate model peformance. Our tasks are:

- (1 point) Split the simulated data into train and test sets
- (4 points) Design and train the model
- (1 point) Print the classificaiton report
- (2 points) Plot the confusion matrix
- (2 points) Plot the per class ROC curves

We have previously created the dataset and stored it to a file. We did this using the `generate_simulated_graphs` and `create_feature_matrix` method as shown in the commented code. We read in the saved data below.

In [ ]:
# methods = {
#     'degree': (nx.degree_centrality, 'node'),
#     'density': (nx.density, 'graph'),
#     'degree_assortativity': (nx.degree_assortativity_coefficient, 'graph'),
#     'diameter': (network_diameter, 'graph'),
#     'betweenness_centrality': (nx.betweenness_centrality, 'node'),
#     'node_clique_number': (nx.node_clique_number, 'node'),
#     'has_bridges': (nx.has_bridges, 'graph')
# }

# # create simulated dataset
# dataset, labels, class_names = generate_simulated_graphs(num_graphs=200, num_nodes=30, p_vals=[0.2, 0.25, 0.3, 0.35])

# # create feature matrix
# X = create_feature_matrix(dataset, methods)

# # save X and labels to disk
# np.save('X.npy', X)
# np.save('labels.npy', labels)

# read the feature matrix and labels from disk
X = np.load('../data/hw1/hw1-X.npy')
y = np.load('../data/hw1/hw1-labels.npy')
class_names = [f'C{_}' for _ in range(len(np.unique(y)))]  # Create class names based on unique labels

# check that the feature matrix and labels have the correct shape
print("Node features:", X.shape)
print("Labels:", y.shape)

### 4(a) Split the dataset
In code cell below, split the dataset into a train and test split of 80% and 20% of the data respectively. The splits should be stratified by class.

In [ ]:
### YOUR CODE HERE ###
X_train, X_test, y_train, y_test = None

In [ ]:
# assertion test
assert X_train.shape[0] == 0.8 * X.shape[0], "Training set should have 80% of the samples"
assert X_test.shape[0] == 0.2 * X.shape[0], "Test set should have 20% of the samples"
assert y_train.shape[0] == X_train.shape[0], "Training labels should match training features"
assert y_test.shape[0] == X_test.shape[0], "Test labels should match test features"

### 4(b) Train a Random Forest Model
In the code cell, apply a grid search with cross validation to select the hyperparameters for a Random Forest classifier. The parameter grid for the grid search should include the following:
- `'n_estimators': [50, 100, 200]`
- `'max_depth': [None, 10, 20, 30]`
- `'criterion': ['gini', 'entropy']`

The cross validation should use 10 folds and use 'accuracy' as the scoring metric. Ensure the cross validation is stratified. Be sure to refit the final model on all the training data. Store the best model (the refit model using the best hyperparameters) in the variable `best_model`.

In [ ]:
### YOUR CODE HERE ###

best_model = None

### 4(c) Print the Classificaiton Report
In the code cell below, get the predictions for the test set using the `best_model` and print the classification report.

In [ ]:
### YOUR CODE HERE ###
y_test_pred = None

### 4(d) Plot the Confusion Matrix 
In the code cell below, plot the confusion matrix for the test set predictions for the `best_model`. You should include a title, x-axis label, and a y-axis label.

In [ ]:
### YOUR CODE HERE ###

### 4(e) Plot the ROC Curves
Finally, in the code cell below, plot the ROC curve for each class. Only create one figure window. All of the ROC curves should be on the same plot. Assign a different color to each curve. Include a title, legend, x-axis label, and y-axis label.

In [ ]:
### YOUR CODE HERE ###